In [1]:
import pandas as pd
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lazypredict.Supervised import LazyClassifier
from imblearn.over_sampling import SMOTE

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.helpers import Helpers

In [2]:
helper = Helpers()
properties = helper.load_properties()

try:
    RANDOM_STATE = properties['models']['random_state']
except KeyError as ke:
    raise KeyError(f"Missing key in properties file: {str(ke)}") from ke

In [3]:
# Apply global settings
helper.set_global_settings()

In [4]:
dataset_dir = helper.root_dir / "datasets"

train_df = pd.read_csv(dataset_dir / "train_data.csv")
train_df.head()

,person_age,is_female,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-0.28,1,3,0.60,1,-0.42,1,-0.29,-0.95,1,0
1,-0.28,0,0,1.23,1,0.35,2,1.21,0.55,0,0
2,1.21,1,1,0.61,2,0.66,3,-0.67,-1.28,1,0
3,0.01,0,1,-0.13,2,-0.31,1,1.52,0.84,0,1
4,1.06,1,0,-0.62,1,-0.67,0,-0.48,0.55,0,0


In [5]:
val_df = pd.read_csv(dataset_dir / "validation_set.csv")
val_df.head()

,person_age,is_female,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-1.41,0,0,1.50,1,-0.67,5,0.29,-1.73,1,0
1,0.27,1,1,-1.19,1,-0.67,1,1.30,-1.62,1,0
2,-1.41,1,1,1.20,1,1.87,5,0.06,-2.15,0,1
3,1.35,1,2,-0.86,2,-0.42,3,-2.05,0.84,0,0
4,-0.28,0,0,-0.05,1,-1.48,2,1.01,-0.11,0,0


In [6]:
target_variable = 'loan_status'

train_df[target_variable].value_counts(normalize=True)

loan_status
0   0.78
1   0.22
Name: proportion, dtype: float64

In [7]:
X_train = train_df.drop(columns=[target_variable])
y_train = train_df[target_variable]

X_val = val_df.drop(columns=[target_variable])
y_val = val_df[target_variable]

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")

Training set size: 29771 samples
Validation set size: 8506 samples


In [8]:
# Calculate distribution before SMOTE
freq_y_train = y_train.value_counts(normalize=True) * 100

# Apply SMOTE to training set
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Calculate distribution after SMOTE
freq_y_train_sm = pd.Series(y_train_sm).value_counts(normalize=True) * 100

# Combine into one dataframe
dist = pd.concat(
    [freq_y_train, freq_y_train_sm], 
    axis=1,
    keys=['Before SMOTE (%)', 'After SMOTE (%)']
).fillna(0).round(2)

# Print markdown table
print(f'Class distribution for {target_variable} (Train set)\n')
print(dist.to_markdown(), '\n')

Class distribution for loan_status (Train set)

|   loan_status |   Before SMOTE (%) |   After SMOTE (%) |
|--------------:|-------------------:|------------------:|
|             0 |              77.74 |                50 |
|             1 |              22.26 |                50 | 



In [9]:
clf = LazyClassifier(
    predictions=True, 
    random_state=RANDOM_STATE
    )

models, predictions = clf.fit(X_train_sm, X_val, y_train_sm, y_val)
models.sort_values(by=['F1 Score', 'Balanced Accuracy', 'ROC AUC', 'Time Taken'], ascending=False)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 23143, number of negative: 23143
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001821 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1299
[LightGBM] [Info] Number of data points in the train set: 46286, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
XGBClassifier,0.92,0.89,0.89,0.92,0.57
LGBMClassifier,0.92,0.89,0.89,0.92,5.51
RandomForestClassifier,0.91,0.89,0.89,0.92,9.20
BaggingClassifier,0.91,0.88,0.88,0.91,2.26
ExtraTreesClassifier,0.91,0.89,0.89,0.91,4.58
DecisionTreeClassifier,0.88,0.85,0.85,0.88,0.40
SVC,0.87,0.88,0.88,0.88,46.36
KNeighborsClassifier,0.86,0.86,0.86,0.87,2.15
AdaBoostClassifier,0.86,0.88,0.88,0.87,2.59


Top 5 models:

1. `XGBClassifier`

2. `LGBMClassifier`

3. `RandomForestClassifier`

4. `BaggingClassifier`

5. `ExtraTreesClassifier`

In [10]:
models_of_interest = [
    'XGBClassifier', 'LGBMClassifier', 'RandomForestClassifier', 'BaggingClassifier', 'ExtraTreesClassifier'
]

for model in models_of_interest:
    print('\t\t',model,'\n')
    print(classification_report(y_val, predictions[model]),'\n')

		 XGBClassifier 

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      6612
           1       0.80      0.85      0.82      1894

    accuracy                           0.92      8506
   macro avg       0.88      0.89      0.88      8506
weighted avg       0.92      0.92      0.92      8506
 

		 LGBMClassifier 

              precision    recall  f1-score   support

           0       0.95      0.94      0.95      6612
           1       0.79      0.84      0.82      1894

    accuracy                           0.92      8506
   macro avg       0.87      0.89      0.88      8506
weighted avg       0.92      0.92      0.92      8506
 

		 RandomForestClassifier 

              precision    recall  f1-score   support

           0       0.96      0.93      0.94      6612
           1       0.78      0.86      0.82      1894

    accuracy                           0.91      8506
   macro avg       0.87      0.89      0.88      8506
wei

In [11]:
# Combine SMOTE resampled features + target
sampled_df = pd.DataFrame(X_train_sm, columns=X_train.columns)
sampled_df[target_variable] = y_train_sm

# Save to CSV
file_path = dataset_dir / "sampled_train_data.csv"
sampled_df.to_csv(file_path, index=False)